# 07 — Song-Level Aggregation: A Live Re-Run That Caught Real Drift

**Hypothesis:** `evaluation/retrieval_diagnostics.py`'s score-distribution check found every facet's
top1-vs-top2 margin is small (typically <0.01) -- with ~14,600 segments and often only a few hundred
per genre, there's usually a long plateau of near-tied single-segment candidates. Mean-pooling a
song's segments into one vector before ranking (`retrieval/song_level_index.py`, the same
aggregation Taste Map/Explore already use for visualization) should smooth that segment-level noise
into a sharper song-level signal -- at the real cost of losing moment-level granularity.

**This notebook does not just verify a number -- it caught a real, live discrepancy while doing so.**
Re-running `scripts/compare_song_level_retrieval.py` (entirely read-only -- it only reports, never
writes to any persisted index) reproduced Sound and Harmony's numbers in `streamlit_app/pages/
1_Methodology.py`'s `SONG_LEVEL_COMPARISON` (§7d) exactly, bit-for-bit. **Vocal, Drums, Bass, and
Instrumental did not match.** §3 below diagnoses why, and §4 states what was actually changed in the
live app as a result -- this is not a hypothetical "here's what you should check," it already
happened, in this same work session, as a direct consequence of running this notebook.

## 1. Setup

No CLAP/Demucs/torch calls happen here -- this notebook only reads facet vectors already computed
and indexed by earlier pipeline stages, via `retrieval/song_level_index.py`'s plain numpy + FAISS.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/oyoai/sonic-explorer.git'
REPO_DIR = '/content/sonic-explorer'


def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')


if os.path.exists(f'{REPO_DIR}/.git'):
    run(['git', '-C', REPO_DIR, 'pull'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('sonic_explorer installed from', REPO_DIR)

## 2. Load the real library and run the real comparison

This reuses `scripts/compare_song_level_retrieval.py`'s exact logic (same functions, same k=10,
sample_size=300, default seed=42) -- not reimplemented, so a result here is directly comparable to
that script's own output and to what `SONG_LEVEL_COMPARISON` in Methodology reports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SonicExplorer')
DB_PATH = DRIVE_ROOT / 'artifacts' / 'sonic_explorer.db'
ARTIFACTS_DIR = DRIVE_ROOT / 'artifacts'

print('DB path:', DB_PATH, '-- exists:', DB_PATH.exists())

In [ ]:
import numpy as np

from sonic_explorer.evaluation.genre_cohesion import genre_cohesion_at_k, song_level_genre_cohesion_at_k
from sonic_explorer.evaluation.retrieval_diagnostics import song_level_score_distribution, top1_score_distribution
from sonic_explorer.facets.registry import default_registry
from sonic_explorer.repository.db import init_db
from sonic_explorer.repository.embedding_repository import EmbeddingRepository
from sonic_explorer.repository.song_repository import SongRepository

K, SAMPLE_SIZE = 10, 300

conn = init_db(str(DB_PATH))
song_repo = SongRepository(conn)
embedding_repo = EmbeddingRepository(conn, artifacts_dir=ARTIFACTS_DIR)
facets = default_registry().names()
for f in facets:
    embedding_repo.load_index(f)

fresh_results = {}
print(f"{'facet':14s} {'seg margin':>11s} {'song margin':>12s} {'seg cohesion':>13s} {'song cohesion':>14s}")
for facet_name in facets:
    seg_scores = top1_score_distribution(song_repo, embedding_repo, facet_name=facet_name, sample_size=SAMPLE_SIZE)
    song_scores = song_level_score_distribution(song_repo, embedding_repo, facet_name=facet_name, sample_size=SAMPLE_SIZE)
    seg_cohesion = genre_cohesion_at_k(song_repo, embedding_repo, facet_name=facet_name, k=K, sample_size=SAMPLE_SIZE)
    song_cohesion = song_level_genre_cohesion_at_k(song_repo, embedding_repo, facet_name=facet_name, k=K, sample_size=SAMPLE_SIZE)

    seg_margin = float(np.mean(seg_scores.top1_top2_margins))
    song_margin = float(np.mean(song_scores.top1_top2_margins))
    fresh_results[facet_name] = {
        'seg_margin': seg_margin, 'song_margin': song_margin,
        'seg_cohesion': seg_cohesion.observed * 100, 'song_cohesion': song_cohesion.observed * 100,
    }
    print(f"{facet_name:14s} {seg_margin:11.4f} {song_margin:12.4f} "
          f"{seg_cohesion.observed * 100:12.1f}% {song_cohesion.observed * 100:13.1f}%")

### Real result

```
facet           seg margin  song margin  seg cohesion  song cohesion
sound               0.0080       0.0185         55.4%          52.5%
harmony             0.0187       0.0326         20.1%          21.8%
vocal               0.0108       0.0147         37.1%          35.4%
drums               0.0084       0.0144         38.4%          36.8%
bass                0.0071       0.0115         23.4%          23.6%
instrumental        0.0114       0.0195         39.6%          42.6%
```

Confirmed deterministic (re-ran twice, bit-identical both times -- `seed=42` genuinely fixes this,
no hidden randomness elsewhere in the pipeline).

## 3. A real discrepancy, diagnosed

Comparing the fresh numbers above against `SONG_LEVEL_COMPARISON` as it existed in Methodology
*before this notebook*:

| facet | seg_cohesion (shipped) | seg_cohesion (fresh) | match? |
|---|---|---|---|
| sound | 55.4% | 55.4% | **exact** |
| harmony | 20.1% | 20.1% | **exact** |
| vocal | 34.4% | 37.1% | no |
| drums | 33.5% | 38.4% | no |
| bass | 25.7% | 23.4% | no |
| instrumental | 38.5% | 39.6% | no |

Same code, same seed, same sample size -- if the underlying data were unchanged, this would be
bit-identical on every facet, the way Sound and Harmony actually are. It is not, for exactly the
four stem-separated facets (Vocal, Drums, Bass, Instrumental), and not for Sound/Harmony (which
don't depend on Demucs stem separation at all). **This is the fingerprint of the stem-facet
reprocessing pass already documented in Results** (a data-quality fix removing near-silent isolated
stems that were being indexed as if meaningful) -- "Sound and Harmony were never affected, since
neither depends on stem separation" is Results' own wording, and it predicts exactly this pattern.

**Conclusion: `SONG_LEVEL_COMPARISON`'s original numbers for the four stem facets were captured
*before* that reprocessing pass, and nobody re-ran this comparison afterward to catch the drift.**
Not a bug in the reprocessing pass itself -- a real gap in the "did anything downstream depend on
the old numbers" follow-through.

## 4. What this changes about the actual conclusion -- not just the numbers

The original write-up said genre-cohesion improved for "5 of 6" facets under song-level
aggregation. Recomputing the deltas from the fresh numbers tells a materially different story.

In [ ]:
for facet_name, r in fresh_results.items():
    margin_ratio = r['song_margin'] / r['seg_margin']
    cohesion_delta = r['song_cohesion'] - r['seg_cohesion']
    print(f"{facet_name:14s} margin_ratio={margin_ratio:.2f}x  cohesion_delta={cohesion_delta:+.1f}pp")

### Real result

```
sound          margin_ratio=2.31x  cohesion_delta=-2.9pp
harmony        margin_ratio=1.74x  cohesion_delta=+1.7pp
vocal          margin_ratio=1.36x  cohesion_delta=-1.7pp
drums          margin_ratio=1.71x  cohesion_delta=-1.6pp
bass           margin_ratio=1.62x  cohesion_delta=+0.2pp
instrumental   margin_ratio=1.71x  cohesion_delta=+3.0pp
```

**Ranking margin still improves for every facet (1.36x-2.31x)** -- that part of the original finding
holds up almost exactly (originally reported as 1.3x-2.3x). **Genre-cohesion's story is genuinely
different now**: only **Instrumental** (+3.0pp) and **Harmony** (+1.7pp) show a real improvement;
**Bass** is roughly flat (+0.2pp, within sampling noise at n=300); **Sound, Vocal, and Drums all now
show a regression** (-2.9pp, -1.7pp, -1.6pp). The original "5 of 6 improved" framing is no longer an
accurate description of the current library.

**This has a real, live consequence worth stating plainly**: `pages/5_Moment_Matcher.py`'s "Match
against: Whole songs" toggle is available for every facet, including Sound, Vocal, and Drums --
three facets where the current evidence shows song-level aggregation now performs *worse* on the
actual task metric, not better. This notebook does not change that live UI (removing or restricting
an option is a product decision, not a documentation fix), but the validation that originally
justified offering it for those three facets is now out of date.

## 5. Production output: what was actually updated as a result

Unlike notebook 06 (verify, change nothing) and more like notebook 05 (a real fix applied), this
investigation produced a real, already-applied change to the shipped app -- **not staged here as a
suggestion, already done in this same work session:**

1. **`streamlit_app/pages/1_Methodology.py`'s `SONG_LEVEL_COMPARISON` dict** now holds the fresh,
   live-verified numbers from §2 above, not the pre-reprocessing-pass ones.
2. **The surrounding narrative text was rewritten**, not just the numbers -- it now states the
   corrected "2 of 6 improved, 1 flat, 3 regressed" finding, explains *why* the story changed (the
   stem-facet reprocessing pass), and explicitly flags the live-UI consequence for Sound/Vocal/Drums
   described in §4, as a real `st.warning()`, not softened into a caption.
3. **A second, independent stale claim was found and fixed in the same pass**: §7d's own text
   claimed song-level aggregation was "not yet wired into Moment Matcher's UI as a selectable
   option." It already is (`pages/5_Moment_Matcher.py`'s "Match against" toggle, confirmed present
   in the actual page code) -- `docs/PROJECT_HISTORY.md`'s own Part 7 confirms this was wired in the
   day after the original investigation, but Methodology's text was never updated to reflect it.

Verified: `ruff check` clean, `tests/test_methodology_page.py` (13 tests) still passes after both
fixes.

## 6. Conclusion

**The headline finding here isn't really about song-level aggregation's merits -- it's that this
exact kind of drift is real and was silently sitting in a "done" page for a while.** A one-time
script result, embedded as a literal for good reasons (no live "before" state to recompute against
in general, per this project's own convention), can quietly go stale when something *else* changes
the data it was measured against -- and nothing else in the pipeline would ever surface that on its
own. The only way to catch it was to actually re-run the comparison, which is the whole reason this
batch of notebooks exists.

**Real, still-open question, explicitly not resolved here:** should the "Whole songs" mode remain
selectable for Sound, Vocal, and Drums in Moment Matcher's UI, now that the evidence for it is
negative for those three facets? That's a product decision (removing a shipped option, or
re-labeling it, or leaving it as a legitimate "sharper ranking, different tradeoff" choice even
without a genre-cohesion win) -- flagged here as a real, current gap between what the UI implies and
what the current data shows, not decided unilaterally by a documentation notebook.